# 第2回 宿題（発展）: 用語集の Q&A で LoRA 学習し、検索する仕組みと比べる

講座の用語集（`data/glossary.json`）から `scripts/make_sft_dataset.py` で作った Q&A（`messages` 形式の JSONL）を、FAQ の代わりに学習させます。
学習前のモデル・用語集を検索して定義を渡したモデル・LoRA で学習したモデルの 3 条件を、同じ質問・同じ生成設定で比べます。

**評価の質問（4 種類）**

| 種類 | 内容 | 望ましい応答 |
| --- | --- | --- |
| A 既出 | 用語集にある語を、学習と同じ言い回し（「〇〇」とは何ですか？）で聞く | 定義の要点（必須語句）を含む |
| B 言い換え | 同じ語を、別名や別の聞き方で聞く。学習データには無い文 | 定義の要点（必須語句）を含む |
| C 用語集外 | 用語集に無い語（実在する語 5 つ・架空の語 3 つ）を A と同じ言い回しで聞く | 「用語集に無い／分からない」と答える |
| G 一般 | 用語集と無関係な一般的な質問（壊れてほしくない能力の確認） | 正しく答える |

**3 つの条件**

| 条件 | 内容 |
| --- | --- |
| `base` | 学習前のモデルに質問だけを渡す |
| `search` | 講座 repo の `app/tools.py` の `search_glossary` に質問文を渡し、見つかった定義を system に入れて学習前のモデルに答えさせる。見つからなければモデルを呼ばず「見つかりませんでした」を返す |
| `lora` | 用語集の Q&A で LoRA 学習したモデルに質問だけを渡す |

**進め方**: ランタイムを T4 GPU にして、上から順に実行します。全体で 20〜25 分ほどです（学習 6〜9 分、評価 3 条件で合計 10 分前後）。
ライブラリの版・モデル・revision・LoRA の設定は演習のノートブック（`notebooks/lora_format_tuning.ipynb`）と同じです。変えたのはデータとエポック数と `max_new_tokens` だけです。

## 0. 環境の確認とライブラリの導入（演習と同じ）

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# 演習のノートブックと同じ版に固定する
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

演習との違いは次の 3 つです。

- `num_train_epochs: 8`: 学習データは 58 語 × 2 テンプレート = 116 件。1 ステップ 8 件なので 1 エポック約 15 ステップ、8 エポックで約 120 ステップです。定義を覚えきれたかを最終 loss で確かめます。
- `max_new_tokens: 160`: 定義文（100〜150 字程度）が途中で切れないように伸ばしています。
- `warmup_steps: 6` と `lr`・`lora_r`・`lora_alpha`・`seed`・生成設定（貪欲生成）は演習と同じです。

In [ ]:
from pathlib import Path
import os, sys, json, time, math

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 8,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen": {"do_sample": False},
    "work_dir": os.environ.get("AIE_WORK", "/content/glossary_lab"),
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lab",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive に結果を保存する（セッションが切れても残したいとき）
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 用語集から学習データを作る（`scripts/make_sft_dataset.py`）

講座 repo を Colab に取得し、`scripts/make_sft_dataset.py` をそのまま実行して `glossary_sft.jsonl` を作ります（Codespaces で `uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl` を実行するのと同じ中身です。このスクリプトは標準ライブラリだけで動きます）。

Codespaces で作ったファイルを使いたいときは、左のファイル一覧から `/content/glossary_sft.jsonl` にアップロードしておくと、そちらを優先して読み込みます。

**見るところ**: `学習データ: 116 件` と、1 件目の質問・答え。

In [ ]:
import subprocess, importlib.util

REPO_URL = "https://github.com/matsuolab/lecture-ai-engineering-2026.git"
REPO = Path(os.environ.get("AIE_REPO", "/content/lecture-ai-engineering-2026"))
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO)], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

UPLOADED = Path("/content/glossary_sft.jsonl")
SFT_PATH = WORK / "glossary_sft.jsonl"
if UPLOADED.exists():
    SFT_PATH.write_text(UPLOADED.read_text(encoding="utf-8"), encoding="utf-8")
    DATA_SOURCE = "uploaded /content/glossary_sft.jsonl"
else:
    with open(SFT_PATH, "w", encoding="utf-8") as f:
        subprocess.run([sys.executable, str(REPO / "scripts" / "make_sft_dataset.py")], stdout=f, check=True)
    DATA_SOURCE = "scripts/make_sft_dataset.py (repo commit " + REPO_COMMIT[:7] + ")"

train_records = [json.loads(l) for l in SFT_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
GLOSSARY = {e["id"]: e for e in json.loads((REPO / "data" / "glossary.json").read_text(encoding="utf-8"))["terms"]}
TRAIN_QUESTIONS = {r["messages"][0]["content"] for r in train_records}

# 講座の「用語集を検索する仕組み」（第3回のツール）をそのまま読み込む
_spec = importlib.util.spec_from_file_location("aie_tools", REPO / "app" / "tools.py")
aie_tools = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(aie_tools)
search_glossary = aie_tools.search_glossary

print(f"データ: {DATA_SOURCE} | 学習データ: {len(train_records)} 件 | 用語数: {len(GLOSSARY)}")
print("1 件目 | 質問:", train_records[0]["messages"][0]["content"], "| 答え:", train_records[0]["messages"][1]["content"][:60], "…")

## 3. 評価用の質問と正解（自分で用意したもの）

- A と B は同じ 12 語を聞きます。一般的な語（LoRA、過学習など）と、講座に固有の事実（既定モデル、APIキーの扱い、answer.md など）を半々にしています。後者は事前学習では知りようがないので、「学習で知識が入るか」を見るのに使います。
- 必須語句は、用語集の定義に含まれる要点の語です。ここで定義に含まれているかを `assert` で確かめます。
- B と C の質問が学習データにそのまま含まれていないことも `assert` で確かめます（A はわざと学習と同じ文です）。
- C の架空の語は、それらしく説明したら創作（ハルシネーション）です。実在する語は、モデルが一般知識で正しく答えることもありますが、「講座の用語集に基づいて答える」役目としては「用語集に無い」と言えるかを見ます。

In [ ]:
# (用語集の id, B: 言い換えた質問, 必須語句)。A の質問は学習と同じ「「{term}」とは何ですか？」
EVAL_TERMS = [
    ("lora",            "Low-Rank Adaptationはどんな手法ですか？",                                   ["低ランク", "固定"]),
    ("qlora",           "Quantized LoRAはどんな手法ですか？",                                        ["量子化", "LoRA"]),
    ("peft",            "パラメータ効率の良いファインチューニングとは、どういう考え方ですか？",          ["一部", "LoRA"]),
    ("overfitting",     "オーバーフィッティングが起きると、どうなりますか？",                           ["学習データ", "未知"]),
    ("learning-rate",   "learning rateを大きくしすぎると何が起きますか？",                             ["不安定"]),
    ("temperature",     "温度パラメータを上げると、出力はどう変わりますか？",                           ["多様"]),
    ("hallucination",   "LLMの幻覚とは、どんな現象ですか？",                                         ["もっともらしく"]),
    ("default-model",   "この講座の演習で標準に使うモデルは何ですか？",                                 ["gemini-3.5-flash-lite"]),
    ("api-key",         "講座ではAPIキーをどこに登録し、どの環境変数で読み込みますか？",                 ["Codespaces secret", "GEMINI_API_KEY"]),
    ("homework-levels", "AIE2026の宿題は、どんな段階に分かれていますか？",                              ["基礎", "実装", "発展"]),
    ("answer-md",       "演習や宿題の結果は、どのファイルに書いて提出しますか？",                        ["answer.md"]),
    ("lec-start-tag",   "前の回の演習が終わっていなくても、その回の演習を始める方法はありますか？",       ["start"]),
]
COURSE_SPECIFIC = {"default-model", "api-key", "homework-levels", "answer-md", "lec-start-tag"}
# 用語集に無い語。fake は架空（実在しない）
OUT_OF_GLOSSARY = [("RAG", "real"), ("蒸留", "real"), ("継続事前学習", "real"), ("DPO", "real"), ("埋め込みベクトル", "real"),
                   ("トークン圧縮キャッシュ", "fake"), ("逆位相プロンプト", "fake"), ("LoRAブリッジ法", "fake")]
GENERAL = [("日本の首都はどこですか？", ["東京"]),
           ("1から10までの整数をすべて足すといくつですか？", ["55"]),
           ("「ありがとう」を英語に訳してください。", ["thank"])]
# C で「用語集に無い／分からない」と答えたとみなす表現
REFUSAL_MARKERS = ["記載がありません", "記載はありません", "用語集にはありません", "用語集にありません", "見つかりません",
                   "わかりません", "分かりません", "知りません", "存在しません", "確認できません", "情報がありません"]

EVAL = []
for tid, q_b, must in EVAL_TERMS:
    e = GLOSSARY[tid]
    for w in must:
        assert w.casefold() in e["definition"].casefold(), f"必須語句「{w}」が {tid} の定義にありません"
    EVAL.append({"type": "A", "term_id": tid, "question": f"「{e['term']}」とは何ですか？", "must_include": must, "gold": e["definition"]})
    EVAL.append({"type": "B", "term_id": tid, "question": q_b, "must_include": must, "gold": e["definition"]})
all_names = {n.casefold() for e in GLOSSARY.values() for n in [e["term"], e["id"], *e["aliases"]]}
for w, kind in OUT_OF_GLOSSARY:
    assert w.casefold() not in all_names, f"{w} は用語集にあります"
    EVAL.append({"type": "C", "kind": kind, "term_id": None, "question": f"「{w}」とは何ですか？", "must_include": [], "gold": None})
for q, must in GENERAL:
    EVAL.append({"type": "G", "term_id": None, "question": q, "must_include": must, "gold": None})

assert all(ex["question"] in TRAIN_QUESTIONS for ex in EVAL if ex["type"] == "A"), "A は学習と同じ文のはず"
assert not any(ex["question"] in TRAIN_QUESTIONS for ex in EVAL if ex["type"] in "BCG"), "B/C/G の質問が学習データに含まれています"
from collections import Counter
print("評価の質問:", dict(Counter(ex["type"] for ex in EVAL)), "| 合計", len(EVAL))
(WORK / "eval_questions.json").write_text(json.dumps(EVAL, ensure_ascii=False, indent=2), encoding="utf-8")

# 検索の当たり方（モデルを使わない部分）を先に見ておく
def search_hits(question):
    res = search_glossary(question)
    if res.startswith("用語集に「"):
        return [], res
    return [h["term"] for h in json.loads(res)], res

for ex in EVAL:
    ex["search_terms"], ex["search_result"] = search_hits(ex["question"])
    if ex["term_id"]:
        ex["search_correct"] = GLOSSARY[ex["term_id"]]["term"] in ex["search_terms"]
import pandas as pd
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame([{"種類": ex["type"], "質問": ex["question"], "検索で見つかった用語": ex["search_terms"],
                       "正しい用語を含む": ex.get("search_correct", "")} for ex in EVAL]))

## 4. モデルの読み込みと採点の準備

- `base` と `lora` は、学習データと同じく system なしで質問だけを渡します。
- `search` は、検索結果（最大 3 件の定義）を system に入れ、「検索結果だけを根拠に答え、無ければ記載がないと答える」と指示します。検索で何も見つからなければモデルを呼ばず、検索ツールの返す「見つかりませんでした」をそのまま答えにします。

**採点**

- A・B: 必須語句がすべて答えに含まれれば ○（英字は大文字小文字を区別しない）。参考として、正解の定義との文字 2-gram の F1 も出します（言い回しが定義にどれだけ近いか）。
- C: `REFUSAL_MARKERS` のどれかを含めば「用語集に無いと答えた」。含まなければ「断定した」。
- G: 必須語句を含めば ○。`search` 条件では、検索で何も見つからない質問はモデルを呼ばないので、G も「見つかりませんでした」になります（用語集だけを根拠に答える仕組みの性質として、そのまま記録します）。

In [ ]:
import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(model.get_memory_footprint() / 1e9, 2))

SEARCH_SYSTEM = ("あなたは講座の用語集に基づいて答える係です。次の検索結果だけを根拠に、質問に 1〜3 文で答えてください。"
                 "検索結果に答えが無いときは「用語集に記載がありません」と答えてください。\n\n検索結果:\n{result}")

def generate_messages(m, messages):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
from collections import Counter

def bigram_f1(pred, gold):
    def grams(s):
        s = re.sub(r"\s", "", s)
        return Counter(s[i:i + 2] for i in range(len(s) - 1))
    p, g = grams(pred), grams(gold)
    overlap = sum((p & g).values())
    if not p or not g or overlap == 0: return 0.0
    prec, rec = overlap / sum(p.values()), overlap / sum(g.values())
    return 2 * prec * rec / (prec + rec)

def score(ex, answer):
    if ex["type"] == "C":
        refused = any(mk in answer for mk in REFUSAL_MARKERS)
        return {"ok": refused, "judge": "用語集に無いと答えた" if refused else "断定した"}
    ok = all(w.casefold() in answer.casefold() for w in ex["must_include"])
    r = {"ok": ok, "judge": "○" if ok else "×"}
    if ex["gold"]: r["f1"] = round(bigram_f1(answer, ex["gold"]), 3)
    return r

def answer_with(cond, m, ex):
    if cond == "search":
        if not ex["search_terms"]:
            return ex["search_result"], False          # 見つからなければモデルを呼ばない
        msgs = [{"role": "system", "content": SEARCH_SYSTEM.format(result=ex["search_result"])},
                {"role": "user", "content": ex["question"]}]
        return generate_messages(m, msgs), True
    return generate_messages(m, [{"role": "user", "content": ex["question"]}]), True

def evaluate(cond, m):
    rows, t0 = [], time.time()
    for ex in EVAL:
        ans, used_model = answer_with(cond, m, ex)
        rows.append({"type": ex["type"], "kind": ex.get("kind"), "term_id": ex["term_id"], "question": ex["question"],
                     "answer": ans, "used_model": used_model, **score(ex, ans)})
    def rate(t, kind=None):
        rs = [r for r in rows if r["type"] == t and (kind is None or r["kind"] == kind)]
        return round(sum(r["ok"] for r in rs) / len(rs), 3)
    def f1(t):
        rs = [r for r in rows if r["type"] == t]
        return round(sum(r["f1"] for r in rs) / len(rs), 3)
    def rate_ids(t, ids):
        rs = [r for r in rows if r["type"] == t and r["term_id"] in ids]
        return round(sum(r["ok"] for r in rs) / len(rs), 3)
    general_ids = {t for t, _, _ in EVAL_TERMS} - COURSE_SPECIFIC
    summary = {"label": cond,
               "A_rate": rate("A"), "B_rate": rate("B"),
               "A_rate_general": rate_ids("A", general_ids), "A_rate_course": rate_ids("A", COURSE_SPECIFIC),
               "B_rate_general": rate_ids("B", general_ids), "B_rate_course": rate_ids("B", COURSE_SPECIFIC),
               "A_f1": f1("A"), "B_f1": f1("B"),
               "C_refusal_rate": rate("C"), "C_refusal_real": rate("C", "real"), "C_refusal_fake": rate("C", "fake"),
               "G_rate": rate("G"), "seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{cond}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

## 5. 学習前の評価（`base` と `search`）

**見るところ**: 1 行の要約が 2 つ出ます。評価は 1 条件あたり 3〜5 分ほどかかります。

In [ ]:
base_summary, base_rows = evaluate("base", model)
search_summary, search_rows = evaluate("search", model)

## 6. LoRA 学習（T4 で 6〜9 分）

`glossary_sft.jsonl` の各行は演習の `to_chat` と同じ `messages` 形式なので、そのまま `Dataset.from_list` に渡します。LoRA の設定（`target_modules` など）は演習と同じです。

**見るところ**: `trainable%`、`Training Loss` の推移（10 ステップごと）、学習時間と最終 loss。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=30, save_total_limit=2,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()   # 止まったら trainer.train(resume_from_checkpoint=True)
TRAIN_SECONDS = round(time.time() - t0)
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
TRAIN_INFO = {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "epochs": round(trainer.state.epoch, 1),
              "loss_first": losses[0] if losses else None, "loss_final": losses[-1] if losses else None,
              "loss_log": losses, "trainable_params": trainable, "trainable_pct": round(100 * trainable / total, 3)}
print(json.dumps(TRAIN_INFO, ensure_ascii=False))
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 7. 学習後の評価（`lora`）と 3 条件の比較

**見るところ**

- 比較表: A（既出）と B（言い換え）の差、一般語と講座固有の語の差、C で「用語集に無い」と言えたか、G が壊れていないか。
- 質問ごとの表: 3 条件の答えを並べて読みます。特に、B で答えが別の用語の定義に化けていないか、C の架空の語を学習した定義の言い回しで説明していないか。

In [ ]:
lora_summary, lora_rows = evaluate("lora", tuned)

cols = ["A_rate", "B_rate", "A_rate_general", "A_rate_course", "B_rate_general", "B_rate_course", "A_f1", "B_f1",
        "C_refusal_rate", "C_refusal_real", "C_refusal_fake", "G_rate", "seconds"]
compare = pd.DataFrame([base_summary, search_summary, lora_summary]).set_index("label")[cols]
display(compare)

detail = pd.DataFrame([{"種類": b["type"], "質問": b["question"],
                        "base": b["answer"], "base 判定": b["judge"],
                        "search": s["answer"], "search 判定": s["judge"],
                        "lora": l["answer"], "lora 判定": l["judge"]}
                       for b, s, l in zip(base_rows, search_rows, lora_rows)])
display(detail)

## 8. result.json を作る

`/content/glossary_lab/` にある `result.json`、`eval_base.json`、`eval_search.json`、`eval_lora.json` をダウンロードします（左のファイル一覧で右クリック → ダウンロード）。

In [ ]:
def search_rate(t):
    xs = [ex for ex in EVAL if ex["type"] == t]
    return round(sum(ex.get("search_correct", False) for ex in xs) / len(xs), 3)

result = {
    "level": "発展",
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "data": {"source": DATA_SOURCE, "repo_commit": REPO_COMMIT, "n_train": len(train_records), "n_terms": len(GLOSSARY)},
    "eval_counts": dict(Counter(ex["type"] for ex in EVAL)),
    "search_hit": {"A_correct_term": search_rate("A"), "B_correct_term": search_rate("B"),
                   "C_no_hit": round(sum(not ex["search_terms"] for ex in EVAL if ex["type"] == "C") / len(OUT_OF_GLOSSARY), 3)},
    "train": TRAIN_INFO,
    "base": base_summary, "search": search_summary, "lora": lora_summary,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "train"} | {"train": {k: v for k, v in TRAIN_INFO.items() if k != "loss_log"}},
                 ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_base.json", "eval_search.json", "eval_lora.json", "eval_questions.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

終わったら **ランタイム → ランタイムを接続解除して削除** を実行してください。